## 🎯 Learning Objectives
* Understand the necessity and benefits of query routing in advanced RAG systems.
* Learn how to implement an LLM-powered query router using LangGraph's conditional edges.
* Differentiate between queries best suited for retrieval and those for direct generation.
* Analyze the performance trade-offs and typical use cases for query routing.


## ADV04-L05: Routing Queries Between Retrieval and Generation

In the evolving landscape of Agentic RAG systems, not all queries are created equal. Some demand factual grounding from a knowledge base (retrieval), while others are best handled by the LLM's inherent generative capabilities (generation). Attempting to retrieve documents for every query, especially those that are purely conversational, creative, or require common knowledge, can introduce unnecessary latency, cost, and even dilute the quality of the LLM's response with irrelevant context.

This is where **query routing** becomes a critical component. Imagine a highly efficient, intelligent librarian at the entrance of a vast library. Instead of sending every patron to the stacks, this librarian quickly assesses their request:

*   "What's the capital of France?" -> Direct answer (generation).
*   "Tell me about the latest research on quantum computing." -> Directs to the science section (retrieval).
*   "Write a poem about a lonely robot." -> Directs to the creative writing desk (generation).

This librarian, much like our query router, makes a swift, informed decision to direct the query to the most appropriate processing path. In an Agentic RAG system, this means deciding whether to invoke a retrieval step, directly generate a response, or even route to a specialized tool.

### Why is Query Routing Essential?

1.  **Efficiency and Cost Savings**: Avoids unnecessary vector database lookups and embedding model calls for queries that don't require external knowledge.
2.  **Improved Accuracy**: Prevents the LLM from being distracted or misled by irrelevant retrieved documents when a direct generative response is more appropriate.
3.  **Reduced Latency**: Bypassing retrieval steps for certain queries can significantly speed up response times.
4.  **Enhanced User Experience**: Provides more direct and relevant answers by applying the right processing strategy.
5.  **Scalability**: Allows for the integration of multiple specialized tools or retrieval sources, with the router acting as a central dispatcher.

LangGraph, with its powerful state management and conditional edges, provides an elegant framework for implementing such intelligent routing. We can leverage the reasoning capabilities of a large language model (LLM) itself to act as our "intelligent librarian," analyzing the incoming query and deciding the optimal next step in our graph. This lesson will demonstrate how to build such a router using LangGraph, directing queries between a simulated retrieval process and a direct generation process.

By 2026, advanced LLMs like Gemini 1.5 Pro, GPT-4o, and Claude 3.5 Sonnet are exceptionally good at classification and instruction following, making them ideal candidates for robust routing decisions. We'll use one of these modern LLMs to power our router.`, 


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install -U langchain langchain_community langgraph langchain_google_genai

import os
from typing import Literal, TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END

# --- Configuration --- #
# Set your Google API key. Replace with your actual key or set as an environment variable.
# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

# Initialize the LLM for routing and generation
# We'll use a powerful model like Gemini 1.5 Pro for its strong reasoning capabilities.
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro-latest", temperature=0.1)

# --- Define Graph State --- #
# The state will hold the query and potentially the retrieved documents and the final answer.
class GraphState(TypedDict):
    query: str
    retrieved_docs: str
    generation: str

# --- Nodes Definition --- #

def retrieve_node(state: GraphState) -> GraphState:
    """Simulates a retrieval step. In a real system, this would query a vector DB."""
    print("---RETRIEVAL NODE--- Called")
    query = state["query"]
    # Simulate retrieving relevant documents based on the query
    # In a real RAG, this would involve embedding the query, searching a vector store,
    # and fetching document content.
    if "latest research" in query.lower() or "technical details" in query.lower():
        retrieved_docs = (
            "Document 1: The latest research in quantum computing focuses on topological qubits and error correction codes. "
            "Document 2: Advanced AI models in 2026 leverage multimodal inputs and self-supervised learning for enhanced generalization. "
            "Document 3: LangGraph enables robust agentic workflows through state management and conditional execution."
        )
    else:
        retrieved_docs = "No specific technical documents found for this query."

    return {"retrieved_docs": retrieved_docs, "query": query}

def generate_node(state: GraphState) -> GraphState:
    """Generates a response, potentially using retrieved documents."""
    print("---GENERATION NODE--- Called")
    query = state["query"]
    retrieved_docs = state.get("retrieved_docs", "")

    if retrieved_docs and retrieved_docs != "No specific technical documents found for this query.":
        # If documents were retrieved, use them for grounded generation
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert assistant. Answer the user's question based *only* on the provided context. If the answer is not in the context, state that you cannot find it."),
            ("human", "Context: {context}\nQuestion: {question}")
        ])
        chain = prompt | llm
        response = chain.invoke({"context": retrieved_docs, "question": query}).content
    else:
        # If no documents or irrelevant documents, generate directly
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful and creative assistant. Answer the user's question directly."),
            ("human", "{question}")
        ])
        chain = prompt | llm
        response = chain.invoke({"question": query}).content

    return {"generation": response, "query": query}


def route_query(state: GraphState) -> Literal["retrieve", "generate"]:
    """Uses an LLM to decide whether to retrieve documents or generate directly."""
    print("---ROUTER NODE--- Called")
    query = state["query"]

    # Prompt the LLM to classify the query
    router_prompt = ChatPromptTemplate.from_messages([
        ("system", 
         "You are a query routing agent. Your task is to analyze the user's query and determine if it requires external knowledge retrieval or if it can be answered directly by a large language model's general knowledge or creative abilities. "
         "Respond with 'retrieve' if the query is factual, specific, or requires up-to-date information that would typically be found in a knowledge base. "
         "Respond with 'generate' if the query is conversational, creative, asks for common knowledge, or can be answered without external lookup. "
         "Your response MUST be either 'retrieve' or 'generate'. Do not include any other text."
        ),
        ("human", "{query}")
    ])

    # Use a simple chain to get the routing decision
    decision = (router_prompt | llm).invoke({"query": query}).content.strip().lower()

    print(f"Router Decision: {decision}")
    if "retrieve" in decision:
        return "retrieve"
    else:
        return "generate"

# --- Build the LangGraph Workflow --- #

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("router", route_query)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

# Set the entry point
workflow.set_entry_point("router")

# Add conditional edges from the router
workflow.add_conditional_edges(
    "router",
    route_query, # The function that determines the next node
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)

# Add edges from retrieve and generate to END
workflow.add_edge("retrieve", "generate") # After retrieval, always go to generation to use the docs
workflow.add_edge("generate", END) # After direct generation, the process ends

# Compile the graph
app = workflow.compile()

# --- Test the Workflow --- #
print("\n--- Testing Queries ---")

# Query 1: Requires retrieval
query1 = "What are the latest advancements in AI models as of 2026?"
print(f"\nQuery 1: {query1}")
for s in app.stream({"query": query1}):
    print(s)

# Query 2: Can be generated directly
query2 = "Write a short, optimistic poem about the future of humanity."
print(f"\nQuery 2: {query2}")
for s in app.stream({"query": query2}):
    print(s)

# Query 3: Factual, but might be common knowledge or require specific lookup
query3 = "Who won the FIFA World Cup in 2022?"
print(f"\nQuery 3: {query3}")
for s in app.stream({"query": query3}):
    print(s)

# Query 4: A more specific technical query
query4 = "Explain the role of conditional edges in LangGraph for building agentic workflows."
print(f"\nQuery 4: {query4}")
for s in app.stream({"query": query4}):
    print(s)

# Query 5: A simple conversational query
query5 = "Tell me a fun fact about cats."
print(f"\nQuery 5: {query5}")
for s in app.stream({"query": query5}):
    print(s)


### Interpreting the Code Output and Performance Considerations

When you run the provided code, you'll observe the `print` statements indicating which nodes are being activated for each query. This clearly demonstrates the routing in action:

*   **Queries routed to `retrieve` (e.g., "latest advancements in AI models", "role of conditional edges in LangGraph")**: You'll see `---ROUTER NODE--- Called` followed by `Router Decision: retrieve`, then `---RETRIEVAL NODE--- Called`, and finally `---GENERATION NODE--- Called`. The `generation` node will then use the `retrieved_docs` (even if simulated) to formulate its answer.
*   **Queries routed to `generate` (e.g., "optimistic poem", "fun fact about cats")**: You'll see `---ROUTER NODE--- Called` followed by `Router Decision: generate`, and then directly `---GENERATION NODE--- Called`. The `retrieval` step is completely bypassed.

Notice how the `generate_node` intelligently adapts: if `retrieved_docs` are present and relevant, it uses a RAG-style prompt; otherwise, it uses a direct generative prompt. This flexibility is key to building robust agentic systems.

### Performance Trade-offs

**Advantages:**

1.  **Optimized Resource Usage**: For queries that don't need retrieval, we save the computational cost and latency of embedding generation, vector database lookups, and potentially large context window usage for irrelevant documents.
2.  **Improved Response Quality**: By preventing the LLM from being exposed to potentially irrelevant or distracting retrieved documents, the quality of direct generative responses can be higher.
3.  **Faster Responses**: Bypassing the retrieval step for appropriate queries leads to quicker overall response times.
4.  **Scalability and Maintainability**: As your system grows to include more specialized tools or retrieval sources, the router acts as a clean abstraction layer, making the system easier to manage and extend.

**Disadvantages:**

1.  **Added Latency for Routing**: The router itself is an LLM call, which introduces a small amount of latency. For extremely low-latency applications, this might be a consideration, though modern LLMs are very fast.
2.  **Routing Errors**: The LLM acting as a router is not infallible. It might occasionally misclassify a query, leading to suboptimal performance (e.g., retrieving for a simple generative query, or generating directly for a query that needed retrieval). Robust prompt engineering and fine-tuning for the router LLM are crucial.
3.  **Increased Complexity**: Introducing a routing layer adds another component to the system, increasing the overall complexity of the graph.

### Typical Use Cases

Query routing is invaluable in a variety of advanced AI applications:

*   **Hybrid RAG Systems**: Seamlessly blending direct LLM generation with knowledge-base retrieval.
*   **Conversational AI and Chatbots**: Distinguishing between factual questions requiring lookup and conversational turns that can be handled generatively or creatively.
*   **Customer Support Agents**: Routing queries to specific knowledge bases (e.g., product manuals, troubleshooting guides) or to a general-purpose generative model for common questions.
*   **Multi-Tool Agents**: When an agent has access to multiple specialized tools (e.g., calculator, code interpreter, web search, internal knowledge base), a router can decide which tool (or combination) is most appropriate for a given query.
*   **Dynamic Content Generation**: For applications that generate content, routing can determine if the content needs to be grounded in specific data or can be purely creative.

By intelligently directing queries, we build more efficient, accurate, and adaptable agentic systems that can handle a wider range of user intents with grace and precision.`, 


### Resources

*   **LangGraph Documentation**: The official documentation is the best place to dive deeper into state graphs, nodes, and conditional edges.
    *   [LangGraph Introduction](https://langchain-ai.github.io/langgraph/)
    *   [LangGraph Conditional Edges](https://langchain-ai.github.io/langgraph/concepts/graphs/#conditional-edges)
*   **LangChain Expression Language (LCEL)**: Understand how to build robust and composable chains.
    *   [LCEL Overview](https://python.langchain.com/docs/expression_language/)
*   **Google Gemini API Documentation**: For details on using Gemini 1.5 Pro and other Google models.
    *   [Gemini API Overview](https://ai.google.dev/docs/gemini_api_overview)
*   **OpenAI API Documentation**: If you prefer using GPT models for routing and generation.
    *   [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
*   **Anthropic Claude API Documentation**: For using Claude models.
    *   [Anthropic API Docs](https://docs.anthropic.com/claude/reference/)
*   **Advanced RAG Techniques**: Explore more sophisticated routing and RAG strategies.
    *   [LangChain RAG Tutorials](https://python.langchain.com/docs/use_cases/question_answering/)
